# Pregunta 3: fauna CCT20 y oclusión aceptada

Notebook final de consulta de resultados V6 guardados y de la revisión independiente de 582 casos.
No entrena, prepara datos científicos ni ejecuta inferencia. Ejecutar desde `entregable/`, junto con `data/`
e `integridad_datos.json`. Requiere Python, pandas e IPython; no importa PyTorch ni Ultralytics.
El notebook y las predicciones V6 originales permanecen intactos. Las salidas de este documento se vuelven a ejecutar.

## Protocolo v6

YOLOv8s preentrenado, 14 clases animales, semilla 42, resolución 640, batch 16, hasta 50 épocas y paciencia 10. AdamW: tasa inicial 3e-4 y weight decay 1e-4. Se conservan los 1.000 fondos oficiales adicionales de v2 y todos los holdouts. V5 corrige la tercera caja de una foto de train según el recuadro rojo proporcionado por el revisor; conserva las otras dos cajas. Conserva las 89 aprobaciones sin exigir notas por fotografía, conforme a la confirmación humana registrada. Compose v6 selecciona configs/wildlife_v6.toml. V6 conserva los píxeles y las etiquetas aprobadas de V5: para JPEG con espacios tras el marcador final, genera una copia sin esos espacios y verifica igualdad exacta de los píxeles decodificados. El comprobador real de Ultralytics valida todas las imágenes y cajas antes de entrenar. cis_val depurado selecciona best.pt. El usuario definió usar el detector existente de Ultralytics y ajustarlo con CCT20. Esa decisión queda en receipts/architecture-review.json, junto con la revisión completa de las 90 fotos. Las tablas y mapas científicos se generan a partir de la ejecución real.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
from collections import Counter

import pandas as pd
from IPython.display import Image, display

delivery = Path.cwd()
if not (delivery / "integridad_datos.json").is_file():
    raise FileNotFoundError("Ejecutar desde la carpeta entregable, junto con data/.")
data = delivery / "data"
integrity = json.loads((delivery / "integridad_datos.json").read_text(encoding="utf-8"))
for name, expected in integrity["sha256"].items():
    assert hashlib.sha256((data / name).read_bytes()).hexdigest() == expected, name
v6 = data / "v6"
output = v6 / "wildlife"

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def show_json(path):
    display(read_json(path))

def show_figure(path, width=900):
    display(Image(filename=str(path), width=width))

print(f"Integridad verificada: {len(integrity['sha256'])} archivos de evidencia.")

## Entorno y CUDA de la ejecución V6

Las constancias siguientes son evidencias guardadas de V6. Consultarlas no inicia operaciones GPU ni
vuelve a validar el entorno local de lectura. La prueba de reanudación histórica no implica equivalencia bit a bit.

In [ ]:
for name in ["doctor_gpu.json", "amp.json"]:
    show_json(v6 / "receipts" / name)

## Integridad y particiones

Se decodifican las imágenes, se registra SHA-256 y se comprueban dimensiones y aspecto respecto a metadatos originales. Las cajas se normalizan con las dimensiones originales. Los JPG se enlazan en un volumen Linux; las imágenes de origen permanecen en solo lectura. Se excluyen animales sin cajas completas y vehículos sin animales. Solo empty explícito genera etiquetas vacías. El train ECCV oficial no contiene empty y todas las fotos vacías de sus diez cámaras estaban reservadas. La ampliación toma 125 secuencias independientes por cada una de ocho cámaras del train de la colección completa, con selección basada en metadatos. No modifica los holdouts. Los negativos tienen resolución original y proceden de cámaras sin positivos añadidos: ambas diferencias deben declararse al interpretar los resultados.


In [ ]:
manifest = read_json(output / "manifest_summary.json")
display({"identity": manifest["identity"], "bbox_units": manifest["bbox_units"], **manifest["integrity"]})
display(pd.DataFrame([{"split": split, "images": row["images"], "backgrounds": row["backgrounds"],
                      "cameras": len(row["cameras"]), "sequences": row["sequences"]}
                     for split, row in manifest["split_summary"].items()]))
display(pd.DataFrame(list(manifest["sequence_overlaps"].items()), columns=["partition_pair", "shared_sequences"]))
display(pd.DataFrame(manifest["class_map"]))
display(manifest["derivation"])

In [ ]:
display(pd.DataFrame(manifest["exclusion_counts"]))
display(pd.DataFrame(list(manifest["train_class_boxes"].items()), columns=["species", "training_boxes"]))
show_figure(output / "class_balance.png")

## Revisión humana antes del entrenamiento V6

Se conservan las 90 fotografías aprobadas por Miguel, incluida la corrección de la tercera caja de ave,
sin modificar las otras dos. Las notas individuales eran opcionales. La migración V6 conserva fotografías
y cajas aprobadas. La decisión de ajustar YOLOv8s existente y el control del cargador se consultan como
constancias históricas; este notebook no modifica revisiones ni galerías.

In [ ]:
labels = pd.read_csv(output / "label_review.csv").fillna("")
assert len(labels) == 90
show_json(output / "label_review_status.json")
show_figure(output / "label_contact_01.png", width=1100)
show_json(v6 / "receipts/architecture-review.json")
show_json(output / "review_migration.json")
display(manifest["jpeg_materialization"])
show_json(v6 / "receipts/yolo-loader-verification.json")

## Smoke aislado: alcance histórico

La prueba pequeña no aporta cifras al informe. El notebook V6 original ya indicaba que sus resultados
no estaban disponibles en esta raíz de salida; no se generan ni se atribuyen cifras nuevas a esa prueba.

In [ ]:
print("Smoke: no incorporado; no interviene en las métricas científicas guardadas.")

## Entrenamiento completo guardado

V6 terminó en 39 épocas y seleccionó el checkpoint de la época 29 mediante validación.
Las curvas y argumentos se conservan; este notebook no contiene una ruta de entrenamiento o inferencia.

In [ ]:
training = pd.read_csv(output / "yolov8s/results.csv")
assert len(training) == 39
print(f"Épocas registradas: {len(training)}; checkpoint seleccionado: época 29 (V6).")
print((output / "yolov8s/args.yaml").read_text(encoding="utf-8"))
show_figure(output / "training_curves.png")

## Cámaras vistas y nuevas

AP COCO a 101 puntos de recall, IoU 0.50:0.95, confianza 0.001, NMS 0.7 y máximo 300 detecciones. AP de una categoría sin referencia queda indefinida. Se conserva la brecha global y se agrega una comparación sobre categorías con soporte en ambos tests. Los intervalos percentiles del 95 % remuestrean secuencias completas con 200 réplicas, semilla 42 y checkpoint fijo. No miden variación entre semillas ni generalización a regiones nuevas.

En V6 quedaron completos los ocho grupos de intervalos, con 200 réplicas cada uno. Tras un límite de memoria, la recuperación usó un proceso nuevo por grupo, conservando la función original, semilla, predicciones y checkpoint; no repitió el entrenamiento.


In [ ]:
display(pd.read_csv(output / "experiments.csv"))
for split in ["cis_test", "trans_test"]:
    metrics = read_json(output / split / "metrics.json")
    display({"split": split, "map50": metrics["map50"], "map50_95": metrics["map50_95"],
             "protocol": metrics["protocol"]})
    display(pd.DataFrame(metrics["per_species"]))
show_json(output / "generalization.json")
uncertainty = read_json(output / "uncertainty.json")
display({key: uncertainty[key] for key in ["global", "size", "limitations"]})

## Tamaño relativo: resultados V6

Pequeño <1 %, mediano 1–5 %, grande >5 % del área de la foto, en ambos tests.
Las cifras se conservan sin cambios. `robustness.json` es histórico: su estado de oclusión corresponde
a la galería completa original; el complemento aceptado se presenta en la sección siguiente.

In [ ]:
robustness = read_json(output / "robustness.json")
for split, groups in robustness["size_by_split"].items():
    display(pd.DataFrame([{"split": split, "group": group, "objects": score["eligible_objects"],
                          "map50": score["map50"], "map50_95": score["map50_95"]}
                         for group, score in groups.items()]))

## Oclusión: 582 casos aceptados

Se usan todos los 18.647 fotogramas de trans_test, incluidos 1.778 vacíos. La AP no usa pesos de muestreo.
Las cajas no revisadas, excluidas, ambiguas o de otro nivel se ignoran como crowd.
Las coincidencias de la misma clase pueden ignorarse; las detecciones no absorbidas aún pueden ser falsos positivos.
La reducción del soporte con estos fotogramas cambia la tarea: la mAP no es comparable directamente con la global V6.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
from collections import Counter

data = Path.cwd() / "data"
integrity = json.loads((data / "integrity.json").read_text(encoding="utf-8"))
for name, expected in integrity["sha256"].items():
    assert hashlib.sha256((data / name).read_bytes()).hexdigest() == expected, name
results = json.loads((data / "results.json").read_text(encoding="utf-8"))
acceptance = json.loads((data / "review_acceptance.json").read_text(encoding="utf-8"))
protocol = json.loads((data / "protocol.json").read_text(encoding="utf-8"))
with (data / "occlusion_review_sample_600.csv").open(encoding="utf-8-sig", newline="") as stream:
    full = list(csv.DictReader(stream))
with (data / "occlusion_review_accepted_582.csv").open(encoding="utf-8-sig", newline="") as stream:
    accepted = list(csv.DictReader(stream))
assert results["signature"] == integrity["signature"]
assert not results["trained_or_ran_inference"]
print("Integridad de archivos verificada; análisis del modelo fijo V6.")

## Decisiones humanas y soporte

Los 18 casos sin nivel se excluyen por instrucción del usuario. Las notas son opcionales bajo esa misma instrucción; no se rellenan. Los ambiguos permanecen en el resumen descriptivo y no se asignan a baja, media o alta.

In [ ]:
excluded = set(acceptance["excluded_annotation_ids"])
assert len(full) == 600 and len(accepted) == 582 and len(excluded) == 18
assert {row["annotation_id"] for row in full if not row["level"]} == excluded
assert accepted == [row for row in full if row["annotation_id"] not in excluded]
assert acceptance["notes_required"] is False and acceptance["human_notes_fabricated"] is False
case_121 = full[120]
assert case_121["annotation_id"] == protocol["selected"][120]["annotation_id"] == "23098"
assert (case_121["level"], case_121["reviewer"], case_121["visual_evidence"]) == ("medium", "Miguel", "")
counts = Counter(row["level"] for row in accepted)
assert counts == {"low": 340, "medium": 69, "high": 48, "ambiguous": 125}
missing_notes = sum(not row["visual_evidence"].strip() for row in accepted)
assert missing_notes == results["review_status"]["accepted_cases_without_notes"] == 398
labels = {"low": "Baja", "medium": "Media", "high": "Alta", "ambiguous": "Ambigua"}
print(f"Aceptados: {len(accepted)}; excluidos: {len(excluded)}; sin nota: {missing_notes}")
for level, label in labels.items():
    print(f"{label}: {counts[level]} ({counts[level] / len(accepted):.1%})")
print("Caso 121: media, Miguel; nota vacía preservada.")

## Métricas e intervalos condicionales

Se conservan todos los fotogramas de `trans_test`, incluidos fondos sin animales, para que sus falsos positivos cuenten. Los objetos sin revisión, excluidos, ambiguos o de otro nivel se ignoran mediante las reglas COCO congeladas. AP no usa los pesos de muestreo. Los intervalos del 95 % remuestrean secuencias completas, con 200 réplicas y semilla 42; mantienen fijo el modelo.

In [ ]:
for level in ("low", "medium", "high"):
    group = results["sample_detection"][level]
    metrics, uncertainty = group["metrics"], group["uncertainty"]
    assert group["signature"] == results["signature"]
    assert metrics["eligible_objects"] == counts[level] >= 30
    assert uncertainty["replicates"] == 200 and uncertainty["seed"] == 42
    assert uncertainty["confidence_level"] == 0.95
    print(f"{labels[level]}: {metrics['eligible_objects']} objetos; {uncertainty['clusters']} secuencias")
    for key in ("map50", "map50_95"):
        bounds = uncertainty["intervals"][key]
        assert metrics[key] is not None and bounds["lower"] is not None and bounds["upper"] is not None
        assert bounds["lower"] <= bounds["upper"] and bounds["defined_replicates"] == 200
        print(f"  {key}: {metrics[key]:.4f}; IC95% [{bounds['lower']:.4f}, {bounds['upper']:.4f}]; réplicas definidas {bounds['defined_replicates']}/200")
display(pd.DataFrame(results["sample_detection"]["low"]["metrics"]["per_species"]).assign(level="low"))

display(pd.DataFrame(results["sample_detection"]["medium"]["metrics"]["per_species"]).assign(level="medium"))

display(pd.DataFrame(results["sample_detection"]["high"]["metrics"]["per_species"]).assign(level="high"))


## Alcance del resumen descriptivo

Las proporciones observadas se refieren a los 582 casos aceptados. Se conservan los pesos originales para describir su distribución, pero las 18 exclusiones son falta de respuesta humana y no constituyen una nueva muestra aleatoria. No se reutiliza la garantía de planificación de 600 casos ni se afirma representatividad poblacional después de las exclusiones.

In [ ]:
descriptive = results["descriptive_occlusion"]
assert descriptive["confidence_intervals_for_population_proportions"] is None
assert descriptive["original_95pct_plus_minus_5pp_planning_guarantee_claimed"] is False
for level, label in labels.items():
    record = descriptive["levels"][level]
    print(f"{label}: observada {record['observed_case_proportion']:.2%}; razón de pesos entre aceptados {record['original_weight_ratio_among_accepted_cases']:.2%}")
print(f"Fracción del peso original excluido: {descriptive['excluded_design_weight_fraction']:.2%}")
print("No se estimaron intervalos poblacionales de proporciones.")

### Comparaciones y límites de oclusión

mAP50:95 baja 6,04 puntos porcentuales de baja a media y 6,31 de baja a alta. Media y alta difieren 0,28 puntos y sus intervalos se superponen ampliamente; no se calculó IC de la diferencia. Las clases con soporte son 13, 10 y 11: composición y soporte difieren. No es un efecto causal aislado. Los IC no incluyen variación de entrenamiento, desacuerdo de anotación ni cámaras futuras.


## Doce casos Grad-CAM

Seis por test: detección correcta a IoU ≥0.5, omisión/clase incorrecta y falsos positivos en fondo, cuando estén disponibles. La selección contrasta soporte raro/frecuente y horarios diurno/nocturno, con alternativas deterministas si falta un tipo de caso. La hora de captura es una aproximación de iluminación que se inspecciona visualmente. Se explica una propuesta y clase antes de NMS en la capa 9. Los mapas no prueban memorización causal del fondo.

La revisión de V6 registra tres mapas planos con rango [0, 0]: cis_test_3, trans_test_3 y trans_test_5. No ofrecen explicación espacial positiva. Varias fotos rotuladas day por metadatos son oscuras y de apariencia infrarroja; ese rótulo no mide iluminación real. Los doce casos originales y la técnica congelada se conservan.


In [ ]:
if (output / 'cam_cases.json').exists():
    display(pd.DataFrame(read_json(output / 'cam_cases.json'))[['split', 'image_id', 'camera_id', 'case', 'class', 'capture_period', 'training_support', 'score']])
for split in ['cis_test', 'trans_test']:
    show_figure(output / f'cam_{split}_grid.png', width=1100)


## Demostración de actividad ecológica

Una cámara nueva elegida por ID, con todos sus fotogramas oficiales, incluidos los excluidos del análisis de cajas. Se requiere fecha de captura válida y una ráfaga completa de al menos dos fotos. Confianza 0.25. Las detecciones no son individuos; las fotografías por especie se cuentan una vez aunque existan varias cajas. Las fechas sin zona no se convierten a Lima.

La demostración V6 contiene 1.853 fotografías de la cámara 7 en 211 fechas: 1.420 con detecciones y once categorías. La tasa predicha de fotos vacías es 23,37 %. La ráfaga mostrada tiene tres fotogramas; no es un conteo de individuos.


In [ ]:
path = output / 'activity.json'
if path.exists():
    activity = read_json(path)
    display({key: activity[key] for key in ['camera_id', 'sequence_id', 'cohort', 'timezone', 'confidence', 'interpretation']})
    display(pd.DataFrame(activity['camera_days']))
    display(pd.DataFrame(activity['species_days']))
    display(pd.DataFrame(activity['sequence_frames']))
else:
    print('Demostración con predicciones finales pendiente.')


## Informe, seis entregables y límites

Se conservan el detector ajustado y sus decisiones, métricas por especie y desbalance, comparación
conocidas/nuevas y doce Grad-CAM, robustez por tamaño y oclusión aceptada, monitoreo ecológico
y discusión responsable. La AP global V6 no es comparable directamente con la AP de la submuestra.
Las detecciones no son individuos; CCT20 no valida despliegue en Perú. No se atribuye aprobación del docente.
El informe principal se compone por separado en ODT y se exporta a PDF; el anexo de oclusión permanece independiente.

In [ ]:
assert len(read_json(output / "cam_cases.json")) == 12
activity = read_json(output / "activity.json")
assert sum(row["photos"] for row in activity["camera_days"]) == 1853
print("Resultados guardados de V6 y complemento de oclusión consultados; sin entrenamiento ni inferencia.")